# 3 — NACA Code Generation

This notebook converts the geometrical blade parameters into NACA 4-digit profile codes.

It reads the sampled geometry table:

$$
\texttt{./csv/prop\_geometrical\_params.csv}
$$

and writes:

$$
\texttt{./csv/naca\_codes.csv}
$$

The output contains one row per propeller configuration:

$$
\texttt{config\_id},\quad
\texttt{naca\_hub},\quad
\texttt{naca\_inner},\quad
\texttt{naca\_mid},\quad
\texttt{naca\_outer}
$$

The `config_id` is preserved exactly so that the NACA table can be joined with the geometry, mass, XFOIL, and QPROP stages of the pipeline.

## 1. Why a hub NACA code is needed

The blade geometry uses three profile stations to define a smooth loft:

$$
r_\mathrm{inner},\quad r_\mathrm{mid},\quad r_\mathrm{outer}
$$

The inner profile is useful for controlling the CAD loft, but it can lie inside the solid hub. Aerodynamically, the blade should start at the outer hub radius:

$$
r_\mathrm{hub}
$$

Therefore, this notebook computes an additional airfoil section at the hub boundary. The hub profile is obtained by evaluating a cubic spline through the inner, middle, and outer airfoil-shape parameters.

The result is a clean separation between:

- CAD construction profiles, which help define the lofted blade shape.
- Aerodynamic profiles, which are used later by XFOIL and QPROP.

## 2. NACA 4-digit convention

A NACA 4-digit profile has the form:

$$
\text{NACA }MPTT
$$

where:

$$
M = \text{maximum camber as percentage of chord}
$$

$$
P = \text{position of maximum camber in tenths of chord}
$$

$$
TT = \text{maximum thickness as percentage of chord}
$$

For example:

$$
\text{NACA }2412
$$

means:

$$
M = 2, \qquad P = 4, \qquad TT = 12
$$

so the profile has 2% camber, maximum camber at 40% chord, and 12% thickness.

## 3. Constants and configuration

All user-adjustable paths, column names, radial locations, interpolation settings, and NACA limits are defined in the next cell. If the upstream geometry generator changes, this is the first block that should be updated.

In [1]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path
import csv

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline


# ============================================================
# File paths
# ============================================================

INPUT_CSV_PATH = Path("./csv/prop_geometrical_params.csv")
OUTPUT_CSV_PATH = Path("./csv/naca_codes.csv")


# ============================================================
# Input geometry columns
# ============================================================

CONFIG_ID_COLUMN = "config_id"
RADIUS_COLUMN = "radius"
MID_RADIAL_POS_COLUMN = "mid radial pos"

INNER_THICKNESS_COLUMN = "inner thickness"
INNER_MAX_POS_COLUMN = "inner max pos"
INNER_CAMBER_COLUMN = "inner camber"

OUTER_THICKNESS_COLUMN = "outer thickness"
OUTER_MAX_POS_COLUMN = "outer max pos"
OUTER_CAMBER_COLUMN = "outer camber"


# ============================================================
# Output columns
# ============================================================

OUTPUT_COLUMNS = [
    "config_id",
    "naca_hub",
    "naca_inner",
    "naca_mid",
    "naca_outer",
]


# ============================================================
# Radial station definition [mm]
# ============================================================

# Inner loft profile radius. This station may be inside the hub.
INNER_PROFILE_RADIUS_MM = 4.5

# Aerodynamic blade start. This must match the physical outer radius of the hub.
HUB_OUTER_RADIUS_MM = 8.25

# The middle station is read as a normalized spanwise coordinate:
#
#     r_mid = mid_radial_pos * radius
#
# The outer station is located at:
#
#     r_outer = radius


# ============================================================
# Interpolation configuration
# ============================================================

# Used to create the middle profile when the geometry table does not contain
# explicit middle NACA-shape sliders.
MID_PROFILE_INTERPOLATION_METHOD = "cubic_smoothstep"

# Used to evaluate the hub profile from the inner / middle / outer profiles.
# A natural cubic spline is used as a practical Python-side approximation of
# the smooth radial variation produced by a lofted CAD blade.
HUB_PROFILE_SPLINE_BC_TYPE = "natural"


# ============================================================
# NACA code limits
# ============================================================

CAMBER_MIN = 0
CAMBER_MAX = 9

MAX_POS_MIN = 0
MAX_POS_MAX = 9

THICKNESS_MIN = 1
THICKNESS_MAX = 99

# Keep codes as strings so that leading zeros are preserved, e.g. "0021".
STORE_NACA_AS_STRING = True

## 4. Load the geometry table

The geometry table is loaded once. Only the columns required to build NACA codes are validated. The row order and `config_id` values are not modified.

In [2]:
if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_CSV_PATH}. "
        "Run the geometrical-parameter generation notebook first."
    )

geometry_df = pd.read_csv(INPUT_CSV_PATH)

required_columns = [
    CONFIG_ID_COLUMN,
    RADIUS_COLUMN,
    MID_RADIAL_POS_COLUMN,
    INNER_THICKNESS_COLUMN,
    INNER_MAX_POS_COLUMN,
    INNER_CAMBER_COLUMN,
    OUTER_THICKNESS_COLUMN,
    OUTER_MAX_POS_COLUMN,
    OUTER_CAMBER_COLUMN,
]

missing_columns = [column for column in required_columns if column not in geometry_df.columns]

if missing_columns:
    raise ValueError(
        "The geometry CSV is missing the following required columns: "
        f"{missing_columns}"
    )

geometry_df = geometry_df.copy()
geometry_df[required_columns].head()

,config_id,radius,mid radial pos,inner thickness,inner max pos,inner camber,outer thickness,outer max pos,outer camber
0,0,67,0.5,17,6,4,12,7,4
1,1,60,0.6,19,5,6,14,5,8
2,2,80,0.4,18,5,5,11,4,2
3,3,66,0.5,22,4,7,13,2,2
4,4,67,0.4,18,4,8,17,5,3


## 5. Helper functions

The geometry file contains inner and outer airfoil-shape parameters directly. The middle profile is reconstructed as a smooth intermediate section.

For a normalized radial coordinate:

$$
s = \frac{r_\mathrm{mid} - r_\mathrm{inner}}{r_\mathrm{outer} - r_\mathrm{inner}}
$$

this notebook uses the cubic smoothstep weight:

$$
w(s) = 3s^2 - 2s^3
$$

and computes:

$$
q_\mathrm{mid} = q_\mathrm{inner} + w(s)\left(q_\mathrm{outer} - q_\mathrm{inner}\right)
$$

where $q$ can be camber, maximum-camber position, or thickness.

Then the hub profile is evaluated using a cubic spline through:

$$
(r_\mathrm{inner}, q_\mathrm{inner}),\quad
(r_\mathrm{mid}, q_\mathrm{mid}),\quad
(r_\mathrm{outer}, q_\mathrm{outer})
$$

at:

$$
r = r_\mathrm{hub}
$$

In [3]:
def cubic_smoothstep(s):
    """Return the cubic smoothstep interpolation weight."""
    s = np.asarray(s, dtype=float)
    s = np.clip(s, 0.0, 1.0)

    return 3.0 * s**2 - 2.0 * s**3


def interpolate_middle_parameter(inner_value, outer_value, r_inner_mm, r_mid_mm, r_outer_mm):
    """Create the middle profile parameter from inner and outer values."""
    denominator = r_outer_mm - r_inner_mm

    if np.any(denominator <= 0):
        raise ValueError("Outer radius must be larger than inner profile radius.")

    s = (r_mid_mm - r_inner_mm) / denominator

    if MID_PROFILE_INTERPOLATION_METHOD == "cubic_smoothstep":
        weight = cubic_smoothstep(s)
    else:
        raise ValueError(
            f"Unknown middle-profile interpolation method: "
            f"{MID_PROFILE_INTERPOLATION_METHOD}"
        )

    return inner_value + weight * (outer_value - inner_value)


def evaluate_hub_parameter(q_inner, q_mid, q_outer, r_inner_mm, r_mid_mm, r_outer_mm, r_hub_mm):
    """Evaluate one airfoil-shape parameter at the hub radius using a cubic spline."""
    x = np.array([r_inner_mm, r_mid_mm, r_outer_mm], dtype=float)
    y = np.array([q_inner, q_mid, q_outer], dtype=float)

    if not np.all(np.diff(x) > 0):
        raise ValueError(
            "Spline stations must be strictly increasing. Check inner, mid, and outer radii."
        )

    if not (x[0] <= r_hub_mm <= x[-1]):
        raise ValueError(
            "Hub radius must lie between the inner and outer profile stations for interpolation."
        )

    spline = CubicSpline(x, y, bc_type=HUB_PROFILE_SPLINE_BC_TYPE)
    return float(spline(r_hub_mm))


def build_naca_code(thickness, camber, max_pos):
    """Build a NACA 4-digit code as a string, preserving leading zeros."""
    camber_digit = int(np.clip(round(float(camber)), CAMBER_MIN, CAMBER_MAX))
    max_pos_digit = int(np.clip(round(float(max_pos)), MAX_POS_MIN, MAX_POS_MAX))
    thickness_digits = int(np.clip(round(float(thickness)), THICKNESS_MIN, THICKNESS_MAX))

    if camber_digit == 0:
        code = f"00{thickness_digits:02d}"
    else:
        code = f"{camber_digit}{max_pos_digit}{thickness_digits:02d}"

    if STORE_NACA_AS_STRING:
        return str(code)

    return int(code)


def is_valid_naca_code(value):
    """Check whether a value is a valid NACA 4-digit code string."""
    code = str(value).strip().zfill(4)

    if len(code) != 4 or not code.isdigit():
        return False

    camber = int(code[0])
    max_pos = int(code[1])
    thickness = int(code[2:])

    return (
        CAMBER_MIN <= camber <= CAMBER_MAX
        and MAX_POS_MIN <= max_pos <= MAX_POS_MAX
        and THICKNESS_MIN <= thickness <= THICKNESS_MAX
    )

## 6. Generate inner, middle, outer, and hub NACA codes

The inner and outer codes are built directly from the sampled geometry values.

The middle code is built from the reconstructed middle airfoil-shape parameters.

The hub code is built by evaluating a cubic spline through the inner, middle, and outer airfoil-shape parameters at the outer hub radius. This gives the equivalent aerodynamic profile at the blade root boundary.

In [4]:
# Physical radial positions
r_inner_mm = float(INNER_PROFILE_RADIUS_MM)
r_outer_mm = geometry_df[RADIUS_COLUMN].astype(float)
r_mid_mm = geometry_df[MID_RADIAL_POS_COLUMN].astype(float) * r_outer_mm
r_hub_mm = float(HUB_OUTER_RADIUS_MM)

# Direct profile parameters
inner_thickness = geometry_df[INNER_THICKNESS_COLUMN].astype(float)
inner_camber = geometry_df[INNER_CAMBER_COLUMN].astype(float)
inner_max_pos = geometry_df[INNER_MAX_POS_COLUMN].astype(float)

outer_thickness = geometry_df[OUTER_THICKNESS_COLUMN].astype(float)
outer_camber = geometry_df[OUTER_CAMBER_COLUMN].astype(float)
outer_max_pos = geometry_df[OUTER_MAX_POS_COLUMN].astype(float)

# Reconstruct middle profile parameters
mid_thickness = interpolate_middle_parameter(
    inner_value=inner_thickness,
    outer_value=outer_thickness,
    r_inner_mm=r_inner_mm,
    r_mid_mm=r_mid_mm,
    r_outer_mm=r_outer_mm,
)

mid_camber = interpolate_middle_parameter(
    inner_value=inner_camber,
    outer_value=outer_camber,
    r_inner_mm=r_inner_mm,
    r_mid_mm=r_mid_mm,
    r_outer_mm=r_outer_mm,
)

mid_max_pos = interpolate_middle_parameter(
    inner_value=inner_max_pos,
    outer_value=outer_max_pos,
    r_inner_mm=r_inner_mm,
    r_mid_mm=r_mid_mm,
    r_outer_mm=r_outer_mm,
)

# Evaluate hub profile parameters from a cubic spline through inner / mid / outer
hub_thickness = []
hub_camber = []
hub_max_pos = []

for row_index in range(len(geometry_df)):
    hub_thickness.append(
        evaluate_hub_parameter(
            q_inner=inner_thickness.iloc[row_index],
            q_mid=mid_thickness.iloc[row_index],
            q_outer=outer_thickness.iloc[row_index],
            r_inner_mm=r_inner_mm,
            r_mid_mm=r_mid_mm.iloc[row_index],
            r_outer_mm=r_outer_mm.iloc[row_index],
            r_hub_mm=r_hub_mm,
        )
    )

    hub_camber.append(
        evaluate_hub_parameter(
            q_inner=inner_camber.iloc[row_index],
            q_mid=mid_camber.iloc[row_index],
            q_outer=outer_camber.iloc[row_index],
            r_inner_mm=r_inner_mm,
            r_mid_mm=r_mid_mm.iloc[row_index],
            r_outer_mm=r_outer_mm.iloc[row_index],
            r_hub_mm=r_hub_mm,
        )
    )

    hub_max_pos.append(
        evaluate_hub_parameter(
            q_inner=inner_max_pos.iloc[row_index],
            q_mid=mid_max_pos.iloc[row_index],
            q_outer=outer_max_pos.iloc[row_index],
            r_inner_mm=r_inner_mm,
            r_mid_mm=r_mid_mm.iloc[row_index],
            r_outer_mm=r_outer_mm.iloc[row_index],
            r_hub_mm=r_hub_mm,
        )
    )

# Build output table
naca_codes_df = pd.DataFrame(
    {
        "config_id": geometry_df[CONFIG_ID_COLUMN].astype(int),
        "naca_hub": [
            build_naca_code(thickness, camber, max_pos)
            for thickness, camber, max_pos in zip(hub_thickness, hub_camber, hub_max_pos)
        ],
        "naca_inner": [
            build_naca_code(thickness, camber, max_pos)
            for thickness, camber, max_pos in zip(inner_thickness, inner_camber, inner_max_pos)
        ],
        "naca_mid": [
            build_naca_code(thickness, camber, max_pos)
            for thickness, camber, max_pos in zip(mid_thickness, mid_camber, mid_max_pos)
        ],
        "naca_outer": [
            build_naca_code(thickness, camber, max_pos)
            for thickness, camber, max_pos in zip(outer_thickness, outer_camber, outer_max_pos)
        ],
    }
)

naca_codes_df = naca_codes_df[OUTPUT_COLUMNS]
naca_codes_df.head()

,config_id,naca_hub,naca_inner,naca_mid,naca_outer
0,0,4617,4617,4615,4712
1,1,6519,6519,7516,8514
2,2,5518,5518,4516,2411
3,3,7421,7422,5318,2213
4,4,8418,8418,7418,3517


## 7. Export CSV

The output directory is created automatically. NACA codes are stored with pandas string dtype and written with quoting so that leading zeros are preserved in the CSV text file.

When reading this CSV in later notebooks, the NACA columns should still be loaded with `dtype=str`.

In [5]:
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

naca_codes_df.to_csv(
    OUTPUT_CSV_PATH,
    index=False,
    quoting=csv.QUOTE_NONNUMERIC,
)

print(f"Exported NACA-code table to: {OUTPUT_CSV_PATH}")
print(f"Columns: {list(naca_codes_df.columns)}")
print(f"Rows: {len(naca_codes_df)}")

Exported NACA-code table to: csv\naca_codes.csv
Columns: ['config_id', 'naca_hub', 'naca_inner', 'naca_mid', 'naca_outer']
Rows: 5000


## 8. Final checks

The final checks verify that:

1. the output schema is exactly the expected schema,
2. the number of rows matches the input geometry table,
3. `config_id` is preserved in the same order,
4. no NACA code is missing,
5. every NACA code is a valid 4-digit string,
6. leading zeros are preserved in the in-memory table,
7. the exported CSV file exists and can be read back using explicit string dtypes.

In [6]:
all_ok = True


def report_check(name, condition):
    """Print a compact pass/fail message."""
    global all_ok

    status = "OK" if bool(condition) else "FAIL"
    print(f"[{status:4}] {name}")

    if not condition:
        all_ok = False


report_check(
    "Output columns match expected schema",
    list(naca_codes_df.columns) == OUTPUT_COLUMNS,
)

report_check(
    "Number of output rows matches input geometry rows",
    len(naca_codes_df) == len(geometry_df),
)

report_check(
    "config_id values are preserved and in the same order",
    naca_codes_df["config_id"].equals(geometry_df[CONFIG_ID_COLUMN].astype(int)),
)

naca_columns = ["naca_hub", "naca_inner", "naca_mid", "naca_outer"]

report_check(
    "No missing NACA codes",
    naca_codes_df[naca_columns].notna().all().all(),
)

valid_codes = naca_codes_df[naca_columns].apply(lambda column: column.map(is_valid_naca_code))

report_check(
    "All NACA codes are valid 4-digit strings",
    valid_codes.all().all(),
)

report_check(
    "All NACA code columns are stored as object/string dtype in memory",
    all(pd.api.types.is_string_dtype(naca_codes_df[column]) for column in naca_columns),
)

report_check(
    "At least one code with leading zeros is preserved in memory, if present",
    all(naca_codes_df[column].astype(str).str.len().eq(4).all() for column in naca_columns),
)

exported_df = pd.read_csv(
    OUTPUT_CSV_PATH,
    dtype={column: str for column in naca_columns},
)

report_check(
    "Exported CSV can be read back with the same shape",
    exported_df.shape == naca_codes_df.shape,
)

report_check(
    "Exported NACA columns remain 4 characters when read with dtype=str",
    all(exported_df[column].astype(str).str.len().eq(4).all() for column in naca_columns),
)

print("-" * 72)
print("ALL CHECKS PASSED ✓" if all_ok else "SOME CHECKS FAILED")
print("-" * 72)

[OK  ] Output columns match expected schema
[OK  ] Number of output rows matches input geometry rows
[OK  ] config_id values are preserved and in the same order
[OK  ] No missing NACA codes
[OK  ] All NACA codes are valid 4-digit strings
[OK  ] All NACA code columns are stored as object/string dtype in memory
[OK  ] At least one code with leading zeros is preserved in memory, if present
[OK  ] Exported CSV can be read back with the same shape
[OK  ] Exported NACA columns remain 4 characters when read with dtype=str
------------------------------------------------------------------------
ALL CHECKS PASSED ✓
------------------------------------------------------------------------
